# Unit 3.3.1 理解文本練習(必修投寄)

兩個任務:①移除停用詞後找段落最高頻詞的頻率;②對兩句做情感分析。
3.3 教的工具:NLTK(tokenize / stopwords / FreqDist)+ VADER。VADER 只支援英文,任務 2 用課程示例句的英文原文;任務 1 的段落是中文,用 jieba 分詞 + 中文停用詞表。

先確保套件與 VADER 字典就緒(macOS 若 SSL 報錯,本格已含 certifi 修正):

In [1]:
import ssl, certifi, nltk
ssl._create_default_https_context = lambda *a, **k: ssl.create_default_context(cafile=certifi.where())
nltk.download('vader_lexicon', quiet=True)
import jieba
print('ready')

ready


## 任務 1:停用詞移除 + 詞頻

**預期**:移除停用詞後最高頻 =「措施」6 次(若停用詞表不濾掉報導動詞「表示」,它也是 6 次並列——最高頻率都是 6)。

In [2]:
import collections, re

para = """新南威尔士州州长格拉迪斯·贝雷吉克莲表示，她正在努力在实施有效的封锁措施与让民众尽可能安全、自由生活之间取得平衡。四周前实施的限制措施原定于7月31日解除，但新州昨日继续报告大量新增新冠病例，新增145例确诊链接到外部网站。 其中51例在传染期内处于社区中，另外25例在传染期的一部分时间内在社区活动，还有11例的隔离状态仍在调查中。
州长在此次疫情期间多次表示，必须将传染期内在社区的病例数降至接近零，才能放宽限制。贝雷吉克莲正在权衡7月31日之后的生活将会是什么样子，但她已明确表示，我们的任务是让市民尽可能安全、自由地生活。州长昨日表示，几天内将就封锁措施的未来做出决定。然而，她坚称，公共卫生建议将在任何决策中起主导作用，她强调自己把民众置于预算之前，把民众置于经济之前。
新州政府将继续推进大规模疫苗接种战略，并考虑为受影响企业提供进一步的财政支持。然而，居家令仍没有结束的迹象，直到病例数开始下降为止。“我们需要一个非常非常严格的封锁，才能把数字压下来。”首席卫生官凯丽·钱特昨日表示。钱特博士明确指出，近期不太可能允许更多工作人员重返办公室。她说：“我们不能让更多人去工作场所，我们不能有更多在工作场所发生的传播事件，否则会推动更多传播链。”
州长昨日暗示，那些效果不佳的限制措施可能会放宽，她表示政府一直在密切监测公共卫生令的结果。“我不想排除这种可能性，有些措施可能会改变。”贝雷吉克莲说，“我们可能需要在某些地区加大力度，而在其他地区放松限制。”至于具体哪些措施可能会改变，目前尚未透露。
周末的反封锁抗议活动对疫情的影响仍不明确链接到外部网站。，但流行病学专家告诉澳大利亚广播公司（ABC），它可能成为一次超级传播事件。政府还将权衡如何最佳利用现有的辉瑞疫苗库存，在高社区传播地区阻止新冠病毒的扩散。"""

stop_cn = set("""的 了 在 是 与 和 让 将 把 但 而 于 对 为 并 还 仍 就 才 会 更 都 很 又 之 其 中 内 上 下 前 后 之间 之后 之前 以及
她 他 我们 自己 这 那 这种 那些 哪些 什么 至于 然而 目前 已 尚未 没有 不 不能 不想 可能 需要 一个 一次 一直 一部分
表示 说 指出 暗示 坚称 强调 告诉""".split())

tokens = [w for w in jieba.lcut(para) if re.match(r'^[一-鿿]{2,}$', w)]

print('未移除停用詞 top5:', collections.Counter(tokens).most_common(5))
filtered = [w for w in tokens if w not in stop_cn]
freq = collections.Counter(filtered)
print('移除停用詞後 top10:')
for w, c in freq.most_common(10):
    print(f'  {w}: {c}')
print()
top, n = freq.most_common(1)[0]
print(f'答案:最高頻詞「{top}」,頻率 = {n}')

Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/pr/ntc3j48x4tn4b4g380jgx58m0000gn/T/jieba.cache
Loading model cost 0.363 seconds.
Prefix dict has been built successfully.


未移除停用詞 top5: [('表示', 6), ('措施', 6), ('我们', 5), ('可能', 5), ('州长', 4)]
移除停用詞後 top10:
  措施: 6
  州长: 4
  封锁: 4
  限制: 4
  昨日: 4
  传播: 4
  贝雷吉克莲: 3
  民众: 3
  生活: 3
  病例: 3

答案:最高頻詞「措施」,頻率 = 6


## 任務 2:VADER 情感分析

兩個中文句是 3.3 所教 VADER 示例句的翻譯;VADER 只支援英文,故用英文原句。

**預期**:句 1 compound ≈ **-0.3412**(負面);句 2 compound ≈ **+0.6364**(整體正面)。

In [3]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
sid = SentimentIntensityAnalyzer()

s1 = "Sentiment analysis has never been good."          # 情感分析从未表现良好。
s2 = "The plot was not great, but the acting was decent and the cinematography was excellent!"
                                                        # 剧本不算精彩,但表演还算不错,摄影则是极其出色!
for s in (s1, s2):
    print(s)
    print('  ', sid.polarity_scores(s))

Sentiment analysis has never been good.
   {'neg': 0.325, 'neu': 0.675, 'pos': 0.0, 'compound': -0.3412}
The plot was not great, but the acting was decent and the cinematography was excellent!
   {'neg': 0.105, 'neu': 0.634, 'pos': 0.261, 'compound': 0.6364}


解讀:
- 句 1:`never ... good` 的否定結構被 VADER 捕捉,neg=0.325、compound=-0.34 → **負面**(句子本身還帶點自嘲:說「情感分析從未表現良好」的句子被情感分析正確判為負面)。
- 句 2:三個子句情感相反(not great 負 / decent 弱正 / excellent! 強正),VADER 對 `but` 轉折會**加重後半句權重**,加上驚嘆號增幅,整體 compound=+0.64 → **正面**。這正是「詞袋式數情緒詞」做不到、VADER 規則(否定、轉折、標點)有價值的地方。

貼文全文見 `notes/Unit2-3_內嵌練習擬答.md` §10;任務 1 的 top10 輸出與任務 2 的兩組分數可截圖附上。